## 1. Preparar
- **requests** → pedir páginas web · **time** → pausas · **json** → guardar datos · **pandas** → tablas
- **BASE** → dirección de la tienda
- **HEADERS** → dice a la web "soy Chrome"


In [ ]:
import requests, time, json
import pandas as pd

BASE = "https://scuffers.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36",
    "Accept": "application/json,text/html;q=0.9,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9",
}


## 2. Instalar curl_cffi
Librería que se conecta **igual que Chrome** para que Shopify no nos bloquee (error 429).
Solo se ejecuta **una vez**.


In [ ]:
%pip install curl_cffi


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Pedir los productos
- `creq.get(...)` → pide `scuffers.com/products.json?limit=250&page=1`
- `status_code` → **200** bien · **429** demasiadas peticiones · **404** no existe
- `r.json()["products"]` → lista de productos
- `len(...)` → cuántos hay


In [ ]:
from curl_cffi import requests as creq

r = creq.get(f"{BASE}/products.json", params={"limit": 250, "page": 1}, impersonate="chrome")
print(r.status_code)
productos = r.json()["products"]
print(len(productos))


200
250


## 4. Paginación
- Pide página a página (`page=1, 2, 3…`) hasta que una venga **vacía**
- `extend` → junta todos los productos en una sola lista
- `time.sleep(3)` → pausa entre páginas para evitar el 429


In [19]:
productos = []
pagina = 1

while True:
    r = creq.get(f"{BASE}/products.json", params={"limit": 250, "page": pagina}, impersonate="chrome")
    print("página", pagina, "->", r.status_code)

    lote = r.json()["products"]
    if len(lote) == 0:        # página vacía = ya no hay más productos
        break

    productos.extend(lote)    # añade los productos de esta página a la lista
    pagina += 1
    time.sleep(3)             # pausa para no volver a recibir el 429

print("Total productos:", len(productos))


página 1 -> 200
página 2 -> 200


KeyboardInterrupt: 

## 5. Guardar los datos originales
- Se guarda el JSON **sin modificar** en `data/raw/` con la fecha en el nombre
- Incluimos la fuente y la fecha de obtención (lo pide el enunciado)
- `ensure_ascii=False` → guarda bien las tildes y la ñ


In [ ]:
from pathlib import Path
from datetime import datetime

carpeta = Path("../data/raw")              # el notebook está en notebook/, así que subimos un nivel
carpeta.mkdir(parents=True, exist_ok=True)  # crea la carpeta si no existe

hoy = datetime.now().strftime("%Y%m%d")
ruta = carpeta / f"scuffers_products_{hoy}.json"

datos = {
    "fuente": f"{BASE}/products.json",
    "fecha_obtencion": datetime.now().isoformat(timespec="seconds"),
    "n_productos": len(productos),
    "products": productos,
}

with open(ruta, "w", encoding="utf-8") as f:
    json.dump(datos, f, ensure_ascii=False, indent=2)

print("Guardado en", ruta)


Guardado en ..\data\raw\scuffers_products_20261119.json


## 6. Pasar el JSON a tablas
- **df_prod** → 1 fila = 1 producto (título, categoría, tags, fecha…)
- **df_var** → 1 fila = 1 talla de un producto (precio, disponible…)
- Se unen por `product_id`
- Todavía **sin limpiar**: se guardan en `data/raw/`


In [ ]:
filas_prod = []
filas_var = []

for p in productos:
    filas_prod.append({
        "product_id": p["id"],
        "titulo": p["title"],
        "handle": p["handle"],
        "product_type": p["product_type"],
        "tags": "|".join(p["tags"]),
        "published_at": p["published_at"],
        "n_tallas": len(p["variants"]),
        "n_imagenes": len(p["images"]),
    })
    for v in p["variants"]:
        filas_var.append({
            "product_id": p["id"],
            "variant_id": v["id"],
            "talla": v["option1"],
            "precio": v["price"],
            "precio_tachado": v["compare_at_price"],
            "disponible": v["available"],
        })

df_prod = pd.DataFrame(filas_prod)
df_var = pd.DataFrame(filas_var)

print(df_prod.shape, df_var.shape)
df_prod.head()


(1698, 8) (8895, 6)


,product_id,titulo,handle,product_type,tags,published_at,n_tallas,n_imagenes
0,10260013482312,GVS Brown Sandals,gvs-brown-sandals,Man Sandals,FOOTWEAR|Gordon Drop,2026-11-19T17:38:55+01:00,10,5
1,10364214051144,Rattler Ecru Wallet,rattler-ecru-wallet,Man Wallet 3,__label: new in|accesorios|accessories|All pro...,2026-11-17T20:58:41+01:00,1,3
2,10364213428552,Rattler Brown Wallet,rattler-brown-wallet,Man Wallet 3,__label: new in|accesorios|accessories|All pro...,2026-11-17T20:58:39+01:00,1,3
3,10364210118984,Rattler Ecru Duffle Bag,rattler-ecru-duffle-bag,Man Duffle,__label: new in|accesorios|accessories|All pro...,2026-11-17T20:58:37+01:00,1,7
4,10364202844488,Balotelli Black Sweatpants,balotelli-black-sweatpants,Man Tech Sporty/pijamero,__label: new in|All products|balotelli black s...,2026-11-17T20:58:36+01:00,6,6


In [20]:
df_prod.to_csv(carpeta / f"scuffers_productos_{hoy}.csv", index=False, encoding="utf-8")
df_var.to_csv(carpeta / f"scuffers_variantes_{hoy}.csv", index=False, encoding="utf-8")
